# Segmentación gold — transaccional + estratégica + potencial

Reemplaza a `notebook.ipynb`. Corre **después** de las 6 verticales (`correr_todo.ipynb` o los orquestadores por separado) — necesita sus Excel de salida.

**La clave de la tabla gold es (`cod_interlocutor`, `periodo`)** — es una tabla de
historial: un mismo cliente puede tener una fila por cada periodo cargado. Por eso:

- **Nunca se toca un periodo que ya existe.** Si `PERIODO` ya tiene filas en gold, el
  notebook se detiene antes de insertar nada — no hay concepto de "actualizar" un periodo
  ya cargado, solo insertar uno nuevo.
- **`MODO`** decide qué pasa con la boca, el transaccional, la estratégica y el potencial de cada cliente:
  - **`completo`**: se inserta el cálculo fresco de este periodo, sin mirar el historial. Un
    segmento vacío queda como **"Por definir"** (en todas las bocas).
  - **`incremental`**: la base es el gold del periodo INMEDIATAMENTE anterior
    (`PERIODO_GOLD_ANTERIOR`), emparejando **solo por `cod_interlocutor`**. La alterna del trax
    (`seg_tabla_alterna_estrategica_transaccional`) es la autoridad para la **boca** y el
    **transaccional**; la estratégica y el potencial solo cambian si estaban "Por definir":
    - Cliente **nuevo** (su código no estaba): se agrega tal cual. Si es "Por definir", queda sin
      estratégica ("Por definir") ni potencial (NULL).
    - Cliente del periodo anterior que hoy **no viene en la alterna: sale de gold** (no se
      arrastra). Cada salida se clasifica por su boca histórica; si alguna no es de una boca no
      segmentable (MA, FA, PI, OT) el notebook se detiene hasta que pongas `SALIDAS_REVISADAS = True`.
    - Cliente que **cambió de grupo de boca** (se compara en vs2: BO, MM y AB cuentan como una, que es
      como las emite el trax): boca, transaccional, estratégica y potencial son los frescos.
    - Cliente con el **mismo grupo de boca**: se conserva el `AB`/`MM` de gold si la alterna dice
      `BO`; el transaccional es el de la alterna; la estratégica y el potencial solo cambian
      si estaban "Por definir" (si ya tenían segmento se conserva todo, incluido
      `flg_potential = False`). Un cliente con segmento que la alterna trae "Por definir" **no
      regresa**: conserva su fila anterior y se lista para revisarlo.

**PA (panadería/pastelería):** para los clientes de boca `PA`, el segmento estratégico (Estadio 1 a 4)
sale **exclusivamente** de la tabla consolidada de estadios (`TABLA_SEGMENTO_PA`), en los dos modos.
Un PA que no está en la tabla queda "Por definir" (en incremental conserva el estadio de gold si lo tenía).
En incremental, un PA de la tabla toma siempre su estadio de la tabla.


In [1]:
%load_ext autoreload
%autoreload 2

MODO = "incremental"  # "completo" | "incremental"
PERIODO = "2026-09-01"  # primer día del mes que se está cargando

# Periodo previo a la segmentación actual (el último periodo ya cargado en la tabla gold).
# Se usa para filtrar TABLA_GOLD, que está particionada por 'periodo' y no permite un
# SELECT * sin filtro. Por defecto se calcula como el mes calendario anterior a PERIODO,
# pero puedes sobreescribirlo manualmente si el último periodo cargado en gold no fue
# el mes inmediato anterior (por ejemplo, si se saltó una corrida mensual).
from datetime import date

_anio, _mes, _ = (int(x) for x in PERIODO.split("-"))
_mes_ant, _anio_ant = (12, _anio - 1) if _mes == 1 else (_mes - 1, _anio)
PERIODO_GOLD_ANTERIOR = date(_anio_ant, _mes_ant, 1).isoformat()  # <-- editar aquí si es necesario

# Incremental: si hay clientes que salen de gold y su boca histórica sí es segmentable (o no aparece en el
# histórico), el notebook se detiene. Revísalos (variable `salidas`) y pon True para continuar.
SALIDAS_REVISADAS = False

# Segmento estratégico de PA: sale EXCLUSIVAMENTE de esta tabla consolidada de estadios (1 a 4).
# Las columnas se detectan por nombre; si no las encuentra, el error lista las que hay y las pones aquí.
TABLA_SEGMENTO_PA = "acpe-dev-uc-sandbox-aa.dev.segmento_estrategico_pa_consolidado"
PA_COL_CLIENTE = "cod_cliente_dex"
PA_COL_SEGMENTO = "seg_estrategico"

PROJECT_BQ = "acpe-dev-uc-ml"
PROJECT_STAGING = "acpe-dev-uc-sandbox-aa"
TABLA_STAGING = "dev.tmp_segmentacion_0"
PROJECT_GOLD = "acpe-prod-dlk-shrd-gld"
TABLA_GOLD = f"{PROJECT_GOLD}.delivery_cliente_gtm.cr_segmentacion"

RUTA_SALIDA_VERTICALES = "../data_potential/output"  # carpeta con los Excel de las 6 verticales + PDM
RUTA_ENCUESTAS = "../data_potential/input"  # carpeta con survey_bodega.csv, survey_pm.csv, etc.
RUTA_RESPALDO = "../data_potential/output"  # dónde guardar el Excel de respaldo (no es la fuente de verdad)

In [2]:
import sys, time
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.config import load_config
from src.salidas import ultimo_archivo
from src.segmentacion_gold import (
    armar_df_final, aplicar_reglas_negocio, aplicar_segmento_pa, cargar_periodo_gold, clasificar_salidas, combinar_potencial,
    detectar_regresiones, detectar_salidas, flag_mercado_potencial, periodo_ya_cargado,
    preparar_carga_periodo, preparar_segmento_pa, resumen_universo, union_encuestas, union_potencial_clientes,
    union_potencial_mercado_pdm, validar_combinaciones, validar_incremental,
)


## 1. Fuentes — Excel de las 6 verticales + PDM

Busca solo el más reciente de cada uno en `RUTA_SALIDA_VERTICALES` (por nombre de
archivo, que empieza con la fecha `AAAAMMDD`) — no hace falta escribir nombres a mano.

In [3]:
salida = Path(RUTA_SALIDA_VERTICALES)
cfg_verticales = load_config()

def _excel_vertical(vertical: str) -> pd.DataFrame:
    ruta = ultimo_archivo(salida, f"*{cfg_verticales[vertical]['salida']}.xlsx")
    print(f"  {vertical:12s} -> {ruta.name}")
    return pd.read_excel(ruta)

print("Buscando el Excel más reciente de cada vertical en", salida)
salidas_vertical = {
    v: _excel_vertical(v) for v in ["bodegas", "panaderia", "gastro", "hoteles", "lavanderias"]
}
df_potencial_clientes = union_potencial_clientes(salidas_vertical)

archivo_pm = ultimo_archivo(salida, f"*{cfg_verticales['pm_mercado']['salida']}.xlsx")
print(f"  {'pm_mercado':12s} -> {archivo_pm.name}")
df_potencial_pm = pd.read_excel(archivo_pm, sheet_name="Potencial", usecols=["mercado_id", "flag_potential"])
ids_mercado = pd.read_excel(archivo_pm, sheet_name="IDs mercado").dropna()
df_potencial_mercado = flag_mercado_potencial(ids_mercado, df_potencial_pm)

archivo_pdm = ultimo_archivo(salida, "*Potencial_final_PDM.xlsx")
print(f"  {'pdm':12s} -> {archivo_pdm.name}")
hojas_pdm = {
    "abarroteros": pd.read_excel(archivo_pdm, sheet_name="Abarroteros"),
    "lavanderos": pd.read_excel(archivo_pdm, sheet_name="Lavanderos"),
    "impulso": pd.read_excel(archivo_pdm, sheet_name="Impulso"),
}
df_potencial_mercado_pdm = union_potencial_mercado_pdm(hojas_pdm)

df_potencial = combinar_potencial(df_potencial_clientes, df_potencial_mercado_pdm)
print(f"\n✓ Potencial: {len(df_potencial):,} clientes (5 verticales + PDM), "
      f"{len(df_potencial_mercado):,} clientes con flg_mercado_potencial")

Buscando el Excel más reciente de cada vertical en ..\data_potential\output
  bodegas      -> 20261020_potential_estimations_bodegas_nuevo.xlsx
  panaderia    -> 20261020_potential_estimations_panaderia_nuevo.xlsx
  gastro       -> 20261020_potential_estimations_gastro_nuevo.xlsx
  hoteles      -> 20261020_potential_estimations_hotel_nuevo.xlsx
  lavanderias  -> 20261020_potential_estimations_lavanderia_nuevo.xlsx
  pm_mercado   -> 20261020_potential_estimations_market_level_mean_seg_pm.xlsx
  pdm          -> 20261020_Potencial_final_PDM.xlsx

✓ Potencial: 36,632 clientes (5 verticales + PDM), 23,554 clientes con flg_mercado_potencial


## 2. Encuestas

In [4]:
encuestas_dir = Path(RUTA_ENCUESTAS)
encuestas_por_vertical = {
    "bodega": pd.read_csv(encuestas_dir / "survey_bodega.csv"),
    "pm": pd.read_csv(encuestas_dir / "survey_pm.csv"),
    "gastro": pd.read_csv(encuestas_dir / "survey_gastro.csv"),
    "panaderia": pd.read_csv(encuestas_dir / "survey_pan.csv"),
    "hotel": pd.read_csv(encuestas_dir / "survey_hotel.csv"),
    "lavanderia": pd.read_csv(encuestas_dir / "survey_lavanderia.csv"),
}
df_encuestas = union_encuestas(encuestas_por_vertical)
print(f"✓ {len(df_encuestas):,} clientes con encuesta")

✓ 144,934 clientes con encuesta


## 3. Transaccional, estratégica y última oficina de venta (BigQuery)

In [11]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_BQ)
print("✓ Conectado a BigQuery")

query_tranx = f"""
SELECT DISTINCT
    SAFE_CAST(REGEXP_REPLACE(cod_interlocutor, r'[^0-9]', '') AS INT64) AS cod_cliente_dex,
    a.des_segmento_transaccional as des_segmentacion_transaccional,
    a.cod_boca_salida,
    CAST(c.cod_cliente_actual AS INT64) AS cod_cliente_alicorp_actual
FROM `acpe-dev-uc-sandbox-aa.dev_segmentacion.seg_tabla_alterna_estrategica_transaccional` a
LEFT JOIN `acpe-prod-dlk-shrd-gld.delivery_modelo_dimensional.cr_cliente_dex` b
    ON a.cod_interlocutor = b.cod_cliente_dex
LEFT JOIN `acpe-prod-dlk-shrd-gld.delivery_modelo_dimensional.cr_cliente_alicorp` c
    ON b.cod_sucursal = c.cod_sucursal
WHERE periodo = '{PERIODO}'
AND a.des_segmento_transaccional IS NOT NULL
"""
df_tranx = client.query(query_tranx).to_dataframe()
print(f"✓ {len(df_tranx):,} clientes transaccionales")

query_estrategico = f"""
SELECT
    SAFE_CAST(REGEXP_REPLACE(cod_interlocutor, r'[^0-9]', '') AS INT64) AS id_interlocutor_origen,
    des_segmento_estrategica AS segmento,
    cod_boca_salida
FROM `acpe-dev-uc-sandbox-aa.dev_segmentacion.seg_tabla_alterna_estrategica_transaccional`
WHERE periodo = '{PERIODO}'
and des_segmento_transaccional IS NOT NULL
"""
df_estrategico = client.query(query_estrategico).to_dataframe()
print(f"✓ {len(df_estrategico):,} clientes estratégicos")

query_last_values = """
SELECT cod_cliente_dex, des_oficina_venta_alicorp, periodo
FROM (
    SELECT cod_cliente_dex, des_oficina_venta_alicorp, periodo,
           ROW_NUMBER() OVER (PARTITION BY cod_cliente_dex ORDER BY periodo DESC) AS rn
    FROM `acpe-prod-dlk-shrd-gld.delivery_reporte_comercial.cr_venta_sell_out`
    WHERE periodo IS NOT NULL
)
WHERE rn = 1
"""
last_values = client.query(query_last_values).to_dataframe()
print(f"✓ {len(last_values):,} últimas oficinas de venta")

C:\Users\asucasacav\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\google\auth\_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


✓ Conectado a BigQuery


C:\Users\asucasacav\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✓ 224,988 clientes transaccionales
✓ 224,994 clientes estratégicos
✓ 341,320 últimas oficinas de venta


## 4. Armar la fila por cliente + reglas de negocio

In [12]:
df_armado = armar_df_final(
    df_tranx, df_estrategico, df_encuestas, df_potencial_mercado, df_potencial, last_values, PERIODO,
)
df_final = aplicar_reglas_negocio(df_armado)  # conservar_sin_asignar=True: no se eliminan clientes
print(f"✓ {len(df_final):,} clientes en el cálculo fresco")

resumen = resumen_universo(df_armado, df_final)
print("  Universo:", resumen)
if resumen["clientes_perdidos"] > 0:
    print(f"  ⚠️  {resumen['clientes_perdidos']:,} clientes de entrada no llegaron a la salida")
if resumen["filas_duplicadas_descartadas"] > 0:
    print(f"  ℹ️  {resumen['filas_duplicadas_descartadas']:,} filas duplicadas descartadas (se conserva el potencial más alto)")

alertas = validar_combinaciones(df_final)
if not alertas.empty:
    print(f"⚠️  {len(alertas):,} filas con (cod_boca_salida, des_segmento_estrategica) fuera de lo esperado:")
    print(alertas.value_counts().head(10))
else:
    print("✓ Todas las combinaciones (boca, segmento) son válidas")

c:\Users\asucasacav\OneDrive - Grupo Alicorp\Documentos\Proyectos\potencial_completo\potencial\src\segmentacion_gold.py:197: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["flg_encuesta"] = df["flg_encuesta"].fillna(False)
c:\Users\asucasacav\OneDrive - Grupo Alicorp\Documentos\Proyectos\potencial_completo\potencial\src\segmentacion_gold.py:201: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["flag_potential"] = df["flag_potential"].fillna(False)
c:\Users\asucasacav\OneDrive - Grupo Alicorp\Documentos\Proyectos\potencial_completo\potencial\src\segment

✓ 217,749 clientes en el cálculo fresco
  Universo: {'filas_entrada': 239457, 'clientes_entrada': 217749, 'filas_duplicadas_descartadas': 21708, 'clientes_salida': 217749, 'clientes_perdidos': 0}
  ℹ️  21,708 filas duplicadas descartadas (se conserva el potencial más alto)
✓ Todas las combinaciones (boca, segmento) son válidas


## 4b. Segmento estratégico de PA — solo de la tabla consolidada de estadios

Los clientes de boca `PA` toman su estadio de `TABLA_SEGMENTO_PA` (lo que traiga la alterna no cuenta). Un PA que no está en la tabla queda "Por definir".

In [ ]:
df_pa_tabla = client.query(f"SELECT * FROM `{TABLA_SEGMENTO_PA}`").to_dataframe()
segmento_pa = preparar_segmento_pa(df_pa_tabla, col_cliente=PA_COL_CLIENTE, col_segmento=PA_COL_SEGMENTO)
print(f"✓ Tabla de PA: {len(df_pa_tabla):,} filas, {len(segmento_pa):,} clientes con estadio")
print(segmento_pa["des_segmento_estrategica"].value_counts().sort_index().to_string())

df_final, resumen_pa = aplicar_segmento_pa(df_final, segmento_pa)
print("\nPA en el cálculo fresco:", resumen_pa)
if resumen_pa["pa_sin_tabla"] > 0:
    print(f"  ⚠️  {resumen_pa['pa_sin_tabla']:,} clientes PA no están en la tabla de estadios: "
          "quedan 'Por definir' (en incremental conservan su estadio de gold, si lo tenían)")

alertas = validar_combinaciones(df_final)
print(f"✓ Combinaciones (boca, segmento) fuera de lo esperado tras aplicar PA: {len(alertas):,}")

## 5. ¿El periodo ya está cargado? Y, en incremental, el periodo anterior

In [ ]:
# 1) ¿este PERIODO ya está cargado en gold? Si sí, se corta acá: nunca se toca un
#    periodo que ya existe (hay que borrarlo a mano primero si de verdad hay que recargarlo).
if periodo_ya_cargado(client, TABLA_GOLD, PERIODO):
    raise RuntimeError(
        f"PERIODO={PERIODO} ya tiene filas en la tabla gold. No se va a insertar nada encima. "
        "Si de verdad hay que recargarlo, borra esas filas primero (a mano, con cuidado) y "
        "volvé a correr este notebook."
    )
print(f"✓ {PERIODO} todavía no está cargado en gold — se puede insertar")

# 2) en modo incremental, la fila de gold del periodo INMEDIATAMENTE anterior (la base
#    del modo incremental) — se filtra exacto por PERIODO_GOLD_ANTERIOR, obligatorio en esta
#    tabla (require_partition_filter).
gold_periodo_anterior = None
if MODO == "incremental":
    query_gold_anterior = f"""
    SELECT * FROM `{TABLA_GOLD}`
    WHERE periodo = DATE('{PERIODO_GOLD_ANTERIOR}')
    """
    gold_periodo_anterior = client.query(query_gold_anterior).to_dataframe()
    print(f"✓ Gold, periodo anterior ({PERIODO_GOLD_ANTERIOR}): {len(gold_periodo_anterior):,} clientes")
    if gold_periodo_anterior.empty:
        print("  ⚠️  no se encontró ningún cliente en ese periodo — revisa PERIODO_GOLD_ANTERIOR "
              "(¿se saltó alguna corrida mensual?); todos los clientes se tratarán como nuevos (cálculo fresco)")

if MODO == "incremental" and not gold_periodo_anterior.empty:
    # 3) Salidas: clientes de gold anterior que hoy no vienen en la alterna. Salen de gold. Se
    #    clasifican por su boca histórica del último mes de la ventana (PERIODO - 1 mes): solo las
    #    de boca no segmentable (MA, FA, PI, OT) son salidas esperadas.
    salidas = detectar_salidas(df_final, gold_periodo_anterior)
    if salidas.empty:
        boca_hist = pd.DataFrame({"cod_cliente_dex": [], "cod_grupo_precio_dex": []})
    else:
        query_boca_hist = f"""
        SELECT SAFE_CAST(cod_cliente_dex AS INT64) AS cod_cliente_dex,
               ARRAY_AGG(cod_grupo_precio_dex IGNORE NULLS ORDER BY periodo DESC, cod_grupo_precio_dex LIMIT 1)[SAFE_OFFSET(0)] AS cod_grupo_precio_dex
        FROM `acpe-prod-dlk-shrd-gld.delivery_modelo_dimensional.cr_cliente_dex_historico`
        WHERE DATE_TRUNC(periodo, MONTH) = DATE_SUB(DATE('{PERIODO}'), INTERVAL 1 MONTH)
        GROUP BY 1
        """
        boca_hist = client.query(query_boca_hist).to_dataframe()
    salidas = clasificar_salidas(salidas, boca_hist)

    val = validar_incremental(df_final, gold_periodo_anterior, salidas, segmento_pa=segmento_pa)
    print("\nValidación incremental:")
    for k, v in val.items():
        print(f"  {k:28s} {v:>10,}")

    if val["regresan_a_por_definir"] > 0:
        regresiones = detectar_regresiones(df_final, gold_periodo_anterior)
        print(f"\n  ⚠️  {len(regresiones):,} clientes con segmento en gold que la alterna trae 'Por definir': "
              "conservan su fila anterior (no regresan). Revisa el trax:")
        print(regresiones.head(10).to_string(index=False))

    a_revisar = salidas[~salidas["esperada"]]
    if not salidas.empty:
        print("\n  Salidas por motivo:")
        print(salidas["motivo"].value_counts().to_string())
    if not a_revisar.empty and not SALIDAS_REVISADAS:
        raise RuntimeError(
            f"{len(a_revisar):,} clientes salen de gold y su boca histórica NO es una boca no segmentable "
            "(MA, FA, PI, OT). Revisa la variable `a_revisar` (columna `motivo`: 'segmentable' o "
            "'sin_historico'); si es correcto que salgan, pon SALIDAS_REVISADAS = True y vuelve a correr."
        )

df_para_cargar = preparar_carga_periodo(
    df_final, MODO, gold_periodo_anterior=gold_periodo_anterior, segmento_pa=segmento_pa,
)
print(f"✓ Filas a insertar en gold para {PERIODO}: {len(df_para_cargar):,}")

## 6. Respaldo en Excel (no es la fuente de verdad — la tabla gold sí lo es)

In [14]:
fecha = time.strftime("%Y%m%d")
ruta_respaldo = Path(RUTA_RESPALDO) / f"{fecha}_segmentacion_{MODO}_{PERIODO}.xlsx"
ruta_respaldo.parent.mkdir(parents=True, exist_ok=True)
df_para_cargar.to_excel(ruta_respaldo, index=False)
print(f"✓ Respaldo guardado en {ruta_respaldo}")

✓ Respaldo guardado en ..\data_potential\output\20261020_segmentacion_incremental_2026-09-01.xlsx


## 7. Insertar en la tabla gold

Un `INSERT` simple (no `MERGE`): con la clave (cliente, periodo), un periodo que todavía
no existe en gold nunca puede chocar con una fila existente.

In [15]:
# Carga en dos pasos (la función hace los dos):
#   1) deja las filas en una tabla temporal del sandbox (ahí sí se pueden crear tablas)
#   2) INSERT INTO gold SELECT ... FROM esa tabla — el mismo mecanismo del notebook
#      original; solo necesita permiso para escribir datos, no para crear tablas en prod.
# Antes de insertar vuelve a comprobar que PERIODO no esté cargado, así que ejecutar
# esta celda dos veces no duplica nada.
TABLA_STAGING_COMPLETA = f"{PROJECT_STAGING}.{TABLA_STAGING}"
insertadas = cargar_periodo_gold(df_para_cargar, client, TABLA_GOLD, TABLA_STAGING_COMPLETA)
print(f"✓ {insertadas if insertadas is not None else len(df_para_cargar):,} filas insertadas en {TABLA_GOLD} para periodo={PERIODO}")

C:\Users\asucasacav\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✓ 218,495 filas insertadas en acpe-prod-dlk-shrd-gld.delivery_cliente_gtm.cr_segmentacion para periodo=2026-09-01
